# Kinetic Energy Rate Regularisation

## Motivation

The soft energy-shell experiment (§3.7 of the companion note) revealed that
penalising **total energy** $E_\ell = T_\ell + V_\ell$ conflicts with language
modelling: $V_\theta$ needs to develop deep potential wells (V very negative)
to encode semantic certainty, and the shell loss fights this directly.

The correct fix targets **kinetic energy only**. Kinetic energy
$T_\ell = \frac{1}{2}m\|v_\ell\|^2$ is the conformal factor of the Jacobi
metric $g_{ij}^{(\ell)} = 2T_\ell m \delta_{ij}$ — it is what LayerNorm resets
at every step, causing the negative $R^2$-magnitude compliance. The damping term
$\gamma$ is supposed to decay $T$ geometrically; LayerNorm interrupts this.

## Loss design

$$\mathcal{L}_{\text{kin}} = \frac{\lambda}{L-1} \sum_{\ell=1}^{L-1}
  \left( \frac{T_{\ell+1}}{T_\ell + \varepsilon} - e^{-\gamma} \right)^2$$

**Why this works:**
- Ratio form: scale-invariant, robust to large $T$ differences across training
- Targets only $T$ (not $V$): $V_\theta$ free to develop deep wells
- Directly enforces the damped geodesic decay rate on the conformal factor
- $\varepsilon = 10^{-6}$: prevents division by zero at early training

## Sweep

| Run | $\lambda$ | LayerNorm | Description |
|-----|-----------|-----------|-------------|
| baseline | 0 | Yes | Control |
| kin-0.01 | 0.01 | Yes | Gentle kinetic regularisation |
| kin-0.1 | 0.1 | Yes | Main candidate |
| kin-1.0 | 1.0 | Yes | Strong kinetic regularisation |
| kin-0.1-noLN | 0.1 | No | Kinetic loss replaces LayerNorm |
| kin-0.1-finetune | 0.1 | Yes | Two-phase: pretrain (λ=0) then fine-tune |

In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────────────
import subprocess, sys, os, gc, math, json, time
from pathlib import Path
from dataclasses import asdict, fields as dc_fields

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    run('pip install -q transformers huggingface_hub pyarrow scipy scikit-learn')
    if not os.path.isdir('semsimula-paper'):
        run('git clone --depth 1 https://github.com/dimitarpg13/semsimula-paper.git')
    REPO = 'semsimula-paper'
else:
    REPO = os.environ.get('SEMSIMULA_PAPER', '.')

ARCH_DIR = os.path.join(REPO, 'notebooks', 'conservative_arch')
for p in [
    ARCH_DIR,
    os.path.join(ARCH_DIR, 'multixi'),
    os.path.join(ARCH_DIR, 'parf'),
    os.path.join(ARCH_DIR, 'energetic_minima'),
    os.path.join(ARCH_DIR, 'sarf_mass_variant'),
    os.path.join(ARCH_DIR, 'scaleup'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else ('mps' if hasattr(torch.backends, 'mps')
          and torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for autograd.grad stability')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_kinetic_rate')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_kinetic_rate'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_CKPTS = DRIVE_ROOT / 'checkpoints'
DRIVE_CKPTS.mkdir(parents=True, exist_ok=True)
print(f'Drive root  : {DRIVE_ROOT}')
print(f'Results dir : {DRIVE_RESULTS}')

In [ ]:
# ── Cell 2: Data (OOM-safe lightweight path) ──────────────────────
from data_module import get_batch, _download_hf_parquet, _resolve_tinystories_shard, _gpt2_tokenize
import pyarrow.parquet as pq

SCRIPTS_DIR = os.path.join(ARCH_DIR, 'scaleup')
LOGFREQ_PATH = os.path.join(SCRIPTS_DIR, 'results', 'logfreq_surprisal_tinystories.npy')

if not os.path.exists(LOGFREQ_PATH):
    print('Computing logfreq surprisal (one-time)...')
    os.makedirs(os.path.dirname(LOGFREQ_PATH), exist_ok=True)
    subprocess.run(
        [sys.executable, os.path.join(SCRIPTS_DIR, 'compute_unigram_frequencies_tinystories.py')],
        cwd=SCRIPTS_DIR, check=True,
    )

DATA_DIR = os.path.join(ARCH_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

train_cache = os.path.join(DATA_DIR, 'tinystories_train_capped.npy')
if os.path.exists(train_cache):
    train_ids = np.load(train_cache)
    print(f'Loaded cached train tokens: {len(train_ids):,}')
else:
    print('Loading training data (capped)...')
    train_fname = _resolve_tinystories_shard('data/train-00000-of-00004')
    tp = _download_hf_parquet('roneneldan/TinyStories', train_fname,
                              'tinystories_train_00000.parquet')
    all_texts = pq.read_table(tp, columns=['text'])['text'].to_pylist()
    n_use = min(len(all_texts), 15_000)
    chunks = []
    for i in range(0, n_use, 3000):
        chunks.append(_gpt2_tokenize('\n\n'.join(all_texts[i:i+3000])))
        del all_texts[i:i+3000]
        if sum(len(c) for c in chunks) >= 6_000_000:
            break
    del all_texts
    train_ids = np.concatenate(chunks)[:6_000_000]
    del chunks
    np.save(train_cache, train_ids)
    print(f'  Cached {len(train_ids):,} train tokens')

val_cache = os.path.join(DATA_DIR, 'tinystories_val_only.npy')
if os.path.exists(val_cache):
    val_ids = np.load(val_cache)
    print(f'Loaded cached val tokens: {len(val_ids):,}')
else:
    val_fname = _resolve_tinystories_shard('data/validation-00000-of-00001')
    vp = _download_hf_parquet('roneneldan/TinyStories', val_fname, 'tinystories_val.parquet')
    val_texts = pq.read_table(vp, columns=['text'])['text'].to_pylist()
    val_ids = _gpt2_tokenize('\n\n'.join(val_texts[:2000]))
    del val_texts
    np.save(val_cache, val_ids)
    print(f'  Cached {len(val_ids):,} val tokens')

gc.collect()
print(f'Train: {len(train_ids):,}  Val: {len(val_ids):,}')

In [ ]:
# ── Cell 3: Model + kinetic-rate integration patch ────────────────
from model_multixi import (
    ScalarPotentialLMSARFMassLNMultiXi,
    SPLMSARFMassLNMultiXiConfig,
)
import types

KIN_EPS = 1e-6  # prevents division by zero in ratio


def integrate_with_kinetic(
    self, x, emb,
    return_trajectory=False, return_xi_trajectory=False,
):
    """Patched integrate() returning per-layer kinetic energies.

    Returns (h_L, traj_h, traj_xi, T_list)
    where T_list[ell] = mean T_ell (differentiable scalar).
    """
    cfg = self.cfg
    h = self._project(emb) if cfg.ln_after_step else emb
    v = torch.zeros_like(h)
    gamma_val, dt = self.gamma, cfg.dt

    m = self.compute_mass(x, emb)
    m_b = m

    traj_h = [h.detach().cpu()] if return_trajectory else None
    traj_xi = [] if return_xi_trajectory else None
    T_list = []   # kinetic energy per layer

    for _ in range(cfg.L):
        xi_input = h.detach() if cfg.causal_force else h
        xis = self.xi_module(xi_input)
        if return_xi_trajectory:
            traj_xi.append(xis.detach().cpu())

        h_in = h
        if not h_in.requires_grad:
            h_in = h_in.requires_grad_(True)

        V_scalar = self.V_theta(xis, h_in).sum()
        (grad_V,) = torch.autograd.grad(
            V_scalar, h_in,
            create_graph=self.training,
            retain_graph=True,
        )
        f = -grad_V
        v = (v + dt * f / m_b) / (1.0 + dt * gamma_val)
        h_new = h_in + dt * v
        if cfg.ln_after_step:
            h_new = self._project(h_new)
        h = h_new

        # Kinetic energy: T = 0.5 * m * ||v||^2, mean over (B, T)
        v_norm2 = (v * v).sum(dim=-1)     # (B, T)
        if isinstance(m_b, torch.Tensor) and m_b.dim() >= 2:
            m_s = m_b.squeeze(-1) if m_b.dim() > 2 else m_b
        else:
            m_s = m_b
        T_ell = (0.5 * m_s * v_norm2).mean()   # scalar, differentiable
        T_list.append(T_ell)

        if return_trajectory:
            traj_h.append(h.detach().cpu())

    return h, traj_h, traj_xi, T_list


def forward_with_kinetic_loss(
    self, x, targets=None,
    return_trajectory=False, return_xi_trajectory=False,
    kin_lambda=0.0,
):
    """Patched forward() with L_kin = kinetic energy rate regularisation."""
    emb = self._embed(x)
    h_L, traj_h, traj_xi, T_list = self.integrate_with_kinetic(
        x, emb,
        return_trajectory=return_trajectory,
        return_xi_trajectory=return_xi_trajectory,
    )
    logits = h_L @ self.E.weight.T

    ce_loss = None
    if targets is not None:
        ce_loss = F.cross_entropy(
            logits.reshape(-1, self.cfg.vocab_size),
            targets.reshape(-1),
        )

    # Kinetic rate loss: penalise deviation from T_{l+1}/T_l = exp(-gamma)
    kin_loss = torch.tensor(0.0, device=x.device)
    if kin_lambda > 0.0 and len(T_list) >= 2:
        gamma_val = self.gamma.item() if hasattr(self.gamma, 'item') else float(self.gamma)
        target_ratio = math.exp(-gamma_val)
        L = len(T_list)
        for ell in range(L - 1):
            T_curr = T_list[ell].detach()    # detach denominator: no grad through anchor
            T_next = T_list[ell + 1]         # grad flows through numerator only
            ratio = T_next / (T_curr + KIN_EPS)
            kin_loss = kin_loss + (ratio - target_ratio) ** 2
        kin_loss = kin_lambda * kin_loss / (L - 1)

    total_loss = None
    if ce_loss is not None:
        total_loss = ce_loss + kin_loss

    # Diagnostics
    self._last_ce_loss = ce_loss.item() if ce_loss is not None else None
    self._last_kin_loss = kin_loss.item()
    self._last_T_list = [t.item() for t in T_list]

    out = [logits, total_loss]
    if return_trajectory:
        out.append(traj_h)
    if return_xi_trajectory:
        out.append(traj_xi)
    return tuple(out) if len(out) > 2 else (out[0], out[1])


def patch_model(model):
    model.integrate_with_kinetic = types.MethodType(integrate_with_kinetic, model)
    model._forward_with_kin = types.MethodType(forward_with_kinetic_loss, model)
    return model


print('Kinetic-rate patch defined.')
print(f'KIN_EPS = {KIN_EPS}')

In [ ]:
# ── Cell 4: Training configuration ────────────────────────────────

TRAIN_STEPS   = 2000
FINETUNE_STEPS = 500    # for the two-phase run
EVAL_INTERVAL = 200
EVAL_ITERS    = 20
LOG_INTERVAL  = 50
BLOCK_SIZE    = 256
LR            = 5e-4
LR_FINETUNE   = 1e-4    # lower LR for fine-tune phase
WEIGHT_DECAY  = 0.01
WARMUP_STEPS  = 100
MODEL_D, MODEL_L, V_HIDDEN, V_DEPTH, XI_CHANNELS = 128, 6, 512, 3, 4

if DEVICE == 'cuda':
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    BATCH_SIZE = 8 if vram_gb >= 30 else (4 if vram_gb >= 14 else 2)
else:
    BATCH_SIZE = 4

# Sweep: (key, kin_lambda, use_layernorm, is_finetune, color)
SWEEP = [
    ('baseline',      0.0,  True,  False, 'tab:blue'),
    ('kin_0.01',      0.01, True,  False, 'tab:cyan'),
    ('kin_0.1',       0.1,  True,  False, 'tab:orange'),
    ('kin_1.0',       1.0,  True,  False, 'tab:green'),
    ('kin_0.1_noLN',  0.1,  False, False, 'tab:red'),
    ('kin_0.1_ft',    0.1,  True,  True,  'tab:purple'),
]

print(f'Training: {TRAIN_STEPS} steps, bs={BATCH_SIZE}, T={BLOCK_SIZE}')
print(f'Model: d={MODEL_D}, L={MODEL_L}')
for key, lam, ln, ft, _ in SWEEP:
    suffix = ' [2-phase: pretrain + fine-tune]' if ft else ''
    print(f'  {key}: λ={lam}, LN={ln}{suffix}')

In [ ]:
# ── Cell 5: Training loop ─────────────────────────────────────────

def lr_schedule(step, lr, warmup, total):
    if step < warmup:
        return lr * (step + 1) / warmup
    progress = (step - warmup) / max(total - warmup, 1)
    return lr * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def free_mem(model=None):
    if model is not None:
        model.cpu()
        del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()


@torch.no_grad()
def evaluate(model, ids, iters, rng_eval):
    model.eval()
    losses = []
    for _ in range(iters):
        xb, yb = get_batch(ids, BATCH_SIZE, BLOCK_SIZE, rng_eval)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def make_model(use_layernorm, seed):
    torch.manual_seed(seed)
    cfg = SPLMSARFMassLNMultiXiConfig(
        d=MODEL_D, max_len=1024,
        v_hidden=V_HIDDEN, v_depth=V_DEPTH, L=MODEL_L,
        init_m=1.0, init_gamma=1.0,
        vocab_size=50257,
        mass_mode='logfreq',
        logfreq_init_alpha=0.1,
        logfreq_path=LOGFREQ_PATH,
        ln_after_step=use_layernorm,
        xi_channels=XI_CHANNELS,
        xi_alpha_inits=[0.0, 0.5, 0.9, 0.99],
        xi_learnable=True,
    )
    model = ScalarPotentialLMSARFMassLNMultiXi(cfg)
    patch_model(model)
    model.to(DEVICE).train()
    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Model: {n:.2f}M params, LN={use_layernorm}')
    return model, cfg


def run_train_loop(model, kin_lambda, steps, lr, warmup, rng_train, rng_eval,
                   train_losses, ce_losses, kin_losses, kin_profiles, val_ppls,
                   step_offset=0):
    """Inner training loop, appends to the provided lists."""
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    t0 = time.time()

    for step in range(steps):
        global_step = step + step_offset
        lr_now = lr_schedule(step, lr, warmup, steps)
        for pg in optimizer.param_groups:
            pg['lr'] = lr_now

        xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng_train)
        x = torch.from_numpy(xb).to(DEVICE)
        y = torch.from_numpy(yb).to(DEVICE)

        _, loss = model._forward_with_kin(x, y, kin_lambda=kin_lambda)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_losses.append(loss.item())
        ce_losses.append(model._last_ce_loss)
        kin_losses.append(model._last_kin_loss)

        if step % LOG_INTERVAL == 0:
            elapsed = time.time() - t0
            ce_str = f'{model._last_ce_loss:.4f}' if model._last_ce_loss else 'N/A'
            # Compute T ratio per layer for logging
            T = model._last_T_list
            ratios = [T[i+1]/(T[i]+KIN_EPS) for i in range(len(T)-1)] if len(T)>1 else []
            ratio_str = '[' + ', '.join(f'{r:.3f}' for r in ratios) + ']'
            print(f'  step {global_step:5d}  ce={ce_str}  '
                  f'kin={model._last_kin_loss:.5f}  '
                  f'T_ratios={ratio_str}  {elapsed:.0f}s')
            kin_profiles.append({'step': global_step, 'T_list': T,
                                  'T_ratios': ratios})

        if (step + 1) % EVAL_INTERVAL == 0 or step == steps - 1:
            val_loss = evaluate(model, val_ids, EVAL_ITERS, rng_eval)
            val_ppl = math.exp(min(val_loss, 20.0))
            val_ppls.append((global_step, val_ppl))
            print(f'  [eval] step {global_step+1}  val_ppl={val_ppl:.2f}')

        del x, y, loss

    del optimizer
    return time.time() - t0


def train_one(key, kin_lambda, use_layernorm, is_finetune, seed=42):
    ckpt_path = DRIVE_CKPTS / f'{key}.pt'
    if ckpt_path.exists():
        print(f'\n  [{key}] Checkpoint exists, skipping.')
        return ckpt_path

    print(f'\n{"═" * 65}')
    print(f'Training: {key}  λ={kin_lambda}  LN={use_layernorm}  finetune={is_finetune}')
    print(f'{"═" * 65}')

    np.random.seed(seed)
    rng_train = np.random.default_rng(seed)
    rng_eval  = np.random.default_rng(seed + 1)

    train_losses, ce_losses, kin_losses, kin_profiles, val_ppls = [], [], [], [], []

    if is_finetune:
        # Phase 1: train without kin loss
        print('  Phase 1: pre-training (λ=0)...')
        model, cfg = make_model(use_layernorm, seed)
        t1 = run_train_loop(model, 0.0, TRAIN_STEPS, LR, WARMUP_STEPS,
                            rng_train, rng_eval,
                            train_losses, ce_losses, kin_losses, kin_profiles, val_ppls,
                            step_offset=0)
        print(f'  Phase 1 done ({t1:.0f}s). Phase 2: fine-tuning (λ={kin_lambda})...')
        # Phase 2: fine-tune with kin loss
        t2 = run_train_loop(model, kin_lambda, FINETUNE_STEPS, LR_FINETUNE, 20,
                            rng_train, rng_eval,
                            train_losses, ce_losses, kin_losses, kin_profiles, val_ppls,
                            step_offset=TRAIN_STEPS)
        elapsed = t1 + t2
    else:
        model, cfg = make_model(use_layernorm, seed)
        elapsed = run_train_loop(model, kin_lambda, TRAIN_STEPS, LR, WARMUP_STEPS,
                                 rng_train, rng_eval,
                                 train_losses, ce_losses, kin_losses, kin_profiles, val_ppls)

    print(f'  Done in {elapsed:.0f}s ({elapsed/60:.1f} min)')
    model.cpu()
    torch.save({
        'config': asdict(cfg),
        'model_state_dict': model.state_dict(),
        'key': key, 'kin_lambda': kin_lambda,
        'use_layernorm': use_layernorm, 'is_finetune': is_finetune,
        'train_losses': train_losses, 'ce_losses': ce_losses,
        'kin_losses': kin_losses, 'kin_profiles': kin_profiles,
        'val_ppls': val_ppls, 'elapsed_s': elapsed,
    }, ckpt_path)
    print(f'  Saved: {ckpt_path}')
    free_mem(model)
    return ckpt_path


ckpt_paths = {}
for key, lam, ln, ft, _ in SWEEP:
    ckpt_paths[key] = train_one(key, lam, ln, ft)

print(f'\nAll {len(SWEEP)} runs complete.')

In [ ]:
# ── Cell 6: Kinetic energy ratio profiles ─────────────────────────
# Key diagnostic: are the T_{ell+1}/T_ell ratios converging to exp(-gamma)?

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flat

for ax, (key, lam, ln, ft, color) in zip(axes, SWEEP):
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    profiles = ckpt['kin_profiles']

    # Plot T_ratio per layer over training steps
    steps_logged = [p['step'] for p in profiles]
    n_layers_minus1 = len(profiles[0]['T_ratios']) if profiles else 0

    for layer_idx in range(n_layers_minus1):
        ratios_over_time = [p['T_ratios'][layer_idx] for p in profiles]
        ax.plot(steps_logged, ratios_over_time, alpha=0.7,
                label=f'L{layer_idx+1}→{layer_idx+2}')

    # Target ratio line
    # Read gamma from checkpoint config (init_gamma=1.0 -> learned)
    # Approximate exp(-gamma) ~ exp(-1) = 0.368 at init
    ax.axhline(math.exp(-1.0), color='k', ls='--', lw=1.0, label='exp(-γ) target')
    ax.set_title(f'{key}\nλ={lam}, LN={ln}', fontsize=9)
    ax.set_xlabel('Step', fontsize=8); ax.set_ylabel('T_{l+1}/T_l', fontsize=8)
    ax.legend(fontsize=6)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Kinetic Energy Ratios T_{ℓ+1}/T_ℓ Over Training', fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(DRIVE_RESULTS / 'kinetic_ratios.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {DRIVE_RESULTS / "kinetic_ratios.png"}')

In [ ]:
# ── Cell 7: Arm 2 Diagnostic — Geodesic Compliance ────────────────

N_EVAL_BATCHES = 3
rng_diag = np.random.default_rng(123)
eval_batches = []
for _ in range(N_EVAL_BATCHES):
    xb, _ = get_batch(val_ids, 4, 128, rng_diag)
    eval_batches.append(torch.tensor(xb))


def load_model_from_ckpt(key):
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    known = {f.name for f in dc_fields(SPLMSARFMassLNMultiXiConfig)}
    cfg = SPLMSARFMassLNMultiXiConfig(
        **{k: v for k, v in ckpt['config'].items() if k in known})
    if hasattr(cfg, 'logfreq_path'):
        cfg.logfreq_path = LOGFREQ_PATH
    model = ScalarPotentialLMSARFMassLNMultiXi(cfg)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    model.to(DEVICE).eval()
    n = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Loaded {key}: {n:.2f}M params')
    return model


def extract_trajectory(model, x):
    with torch.enable_grad():
        out = model(x, targets=None, return_trajectory=True,
                    return_xi_trajectory=False)
        logits, _loss, traj = out[0], out[1], out[2]
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    traj_cpu = [h.detach().cpu() for h in traj]
    del traj, out, logits
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return traj_cpu


def compute_grad_V(model, h):
    h_in = h.detach().requires_grad_(True)
    xis = model.xi_module(h_in.detach())
    V = model.V_theta(xis, h_in)
    return torch.autograd.grad(V.sum(), h_in, create_graph=False)[0].detach()


def get_mass(model, x):
    emb = model._embed(x)
    m = model.compute_mass(x, emb)
    return m.detach().cpu() if isinstance(m, torch.Tensor) else m


print('═' * 60)
print('ARM 2: Geodesic Compliance')
print('═' * 60)

arm2_all = {}

for key, lam, ln, ft, color in SWEEP:
    print(f'\n── {key} ──')
    model = load_model_from_ckpt(key)
    gamma = model.gamma.item() if hasattr(model.gamma, 'item') else float(model.gamma)

    comp_und, comp_dmp, cos_und, cos_dmp = None, None, None, None

    for bi in range(N_EVAL_BATCHES):
        x = eval_batches[bi].to(DEVICE)
        traj = extract_trajectory(model, x)
        L = len(traj) - 1

        if comp_und is None:
            comp_und = [[] for _ in range(L - 1)]
            comp_dmp = [[] for _ in range(L - 1)]
            cos_und  = [[] for _ in range(L - 1)]
            cos_dmp  = [[] for _ in range(L - 1)]

        with torch.no_grad():
            m = get_mass(model, x)
            m_flat = m.squeeze(-1) if isinstance(m, torch.Tensor) and m.dim() > 2 else m
        del x

        for ell in range(1, L):
            h_prev = traj[ell - 1].to(DEVICE)
            h_curr = traj[ell].to(DEVICE)
            h_next = traj[ell + 1].to(DEVICE)
            v = h_curr - h_prev
            a_obs = h_next - 2 * h_curr + h_prev
            del h_prev, h_next
            grad_V = compute_grad_V(model, h_curr)
            v_norm2 = (v ** 2).sum(dim=-1, keepdim=True)
            mf = (m_flat.unsqueeze(-1).to(DEVICE)
                  if isinstance(m_flat, torch.Tensor) and m_flat.dim() == 2 else m_flat)
            denom = (2.0 * 0.5 * mf * v_norm2).clamp(min=1e-8)
            gV_dot_v = (grad_V * v).sum(dim=-1, keepdim=True)
            a_j = (2.0 * v * gV_dot_v - v_norm2 * grad_V) / denom
            a_d = a_j - gamma * v
            a2 = (a_obs ** 2).sum(dim=-1).mean().item()
            comp_und[ell-1].append(1.0 - ((a_obs - a_j) ** 2).sum(dim=-1).mean().item() / max(a2, 1e-12))
            comp_dmp[ell-1].append(1.0 - ((a_obs - a_d) ** 2).sum(dim=-1).mean().item() / max(a2, 1e-12))
            af = a_obs.reshape(-1, a_obs.shape[-1])
            cos_und[ell-1].append(F.cosine_similarity(af, a_j.reshape(-1, a_j.shape[-1]), dim=-1).mean().item())
            cos_dmp[ell-1].append(F.cosine_similarity(af, a_d.reshape(-1, a_d.shape[-1]), dim=-1).mean().item())
            del h_curr, v, a_obs, grad_V, a_j, a_d

        del traj
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    arm2_all[key] = {
        'r2_und': [np.mean(l) for l in comp_und],
        'r2_dmp': [np.mean(l) for l in comp_dmp],
        'cos_und': [np.mean(l) for l in cos_und],
        'cos_dmp': [np.mean(l) for l in cos_dmp],
        'r2_und_mean': float(np.mean([np.mean(l) for l in comp_und])),
        'r2_dmp_mean': float(np.mean([np.mean(l) for l in comp_dmp])),
        'cos_und_mean': float(np.mean([np.mean(l) for l in cos_und])),
        'cos_dmp_mean': float(np.mean([np.mean(l) for l in cos_dmp])),
    }
    a2 = arm2_all[key]
    print(f'  R²(dmp)={a2["r2_dmp_mean"]:.4f}  cos(dmp)={a2["cos_dmp_mean"]:.4f}  '
          f'R²(und)={a2["r2_und_mean"]:.4f}  cos(und)={a2["cos_und_mean"]:.4f}')
    free_mem(model); del model

print('\nArm 2 complete. ✓')

In [ ]:
# ── Cell 8: Results plots + summary table ─────────────────────────

# Arm 2 compliance plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for key, _, _, _, color in SWEEP:
    a2 = arm2_all[key]
    layers = range(1, len(a2['cos_dmp']) + 1)
    axes[0].plot(layers, a2['cos_dmp'], 'o-', label=key, color=color, markersize=4)
    axes[1].plot(layers, a2['r2_dmp'],  's-', label=key, color=color, markersize=4)
axes[0].set_title('Cosine Compliance (Damped)'); axes[0].set_ylabel('Cosine')
axes[1].set_title('R² Magnitude Compliance (Damped)'); axes[1].set_ylabel('R²')
axes[1].axhline(0, color='gray', ls=':', lw=0.8)
for ax in axes:
    ax.set_xlabel('Layer'); ax.legend(fontsize=7)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('Kinetic Rate Regularisation: Arm 2 Geodesic Compliance',
             fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(DRIVE_RESULTS / 'arm2_kinetic.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Training loss comparison
fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
for key, _, _, _, color in SWEEP:
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    ce = np.array([c for c in ckpt['ce_losses'] if c is not None])
    kl = np.array(ckpt['kin_losses'])
    w = min(50, max(1, len(ce) // 4))
    axes2[0].plot(range(w-1, len(ce)),
                  np.convolve(ce, np.ones(w)/w, mode='valid'),
                  label=key, color=color, alpha=0.8)
    if kl.max() > 0:
        axes2[1].plot(range(w-1, len(kl)),
                      np.convolve(kl, np.ones(w)/w, mode='valid'),
                      label=key, color=color, alpha=0.8)
    vp = ckpt['val_ppls']
axes2[0].set_title('CE Loss'); axes2[0].set_ylabel('Loss')
axes2[1].set_title('Kinetic Rate Loss'); axes2[1].set_ylabel('L_kin')
for ax in axes2:
    ax.set_xlabel('Step'); ax.legend(fontsize=7)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('Training Dynamics', fontweight='bold', y=1.02)
plt.tight_layout()
fig2.savefig(DRIVE_RESULTS / 'training_kinetic.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# Summary table
final_ppls = {}
for key, _, _, _, _ in SWEEP:
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    vp = ckpt['val_ppls']
    final_ppls[key] = vp[-1][1] if vp else float('nan')

results = {
    'config': {'model_d': MODEL_D, 'model_L': MODEL_L, 'train_steps': TRAIN_STEPS,
               'batch_size': BATCH_SIZE, 'block_size': BLOCK_SIZE},
    'sweep': {}
}
for key, lam, ln, ft, _ in SWEEP:
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    results['sweep'][key] = {
        'kin_lambda': lam, 'use_layernorm': ln, 'is_finetune': ft,
        'final_val_ppl': final_ppls[key],
        'final_kin_loss': ckpt['kin_losses'][-1] if ckpt['kin_losses'] else None,
        'final_T_list': ckpt['kin_profiles'][-1]['T_list'] if ckpt['kin_profiles'] else None,
        'final_T_ratios': ckpt['kin_profiles'][-1]['T_ratios'] if ckpt['kin_profiles'] else None,
        'arm2': arm2_all.get(key, {}),
    }

report_path = DRIVE_RESULTS / 'kinetic_rate_report.json'
with open(report_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'\nFull results: {report_path}')

print('\n' + '═' * 90)
print('KINETIC RATE REGULARISATION: SUMMARY')
print('═' * 90)
print(f'{"Config":<20} {"λ":>6} {"LN":>5} {"FT":>4} {"PPL":>8} {"cos(dmp)":>10} {"R²(dmp)":>10} {"L_kin":>10}')
print('-' * 90)
for key, lam, ln, ft, _ in SWEEP:
    a2 = arm2_all.get(key, {})
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    kl = ckpt['kin_losses'][-1] if ckpt['kin_losses'] else 0
    print(f'{key:<20} {lam:>6.2f} {str(ln):>5} {str(ft):>4} '
          f'{final_ppls[key]:>8.1f} '
          f'{a2.get("cos_dmp_mean", float("nan")):>10.4f} '
          f'{a2.get("r2_dmp_mean", float("nan")):>10.4f} '
          f'{kl:>10.6f}')
print('═' * 90)
print('\nKey question: does any λ improve R²(dmp) over baseline without degrading PPL?')
print('\n✓ Experiment complete.')